# Colorado climate stations (daily) — pipeline

Thin orchestration of the four stages (**retrieve → audit → cleanup → publish**) over the CO DWR/CDSS climate daily API. All data logic lives in the `climate_stations` package; this notebook is its human-facing narrative. The headless twin is `python -m climate_stations.pipeline`. Implements issue [#44](https://github.com/CUPIDS-Lab/co-environmental-data/issues/44).

> A CDSS API key is required in practice (anonymous limits since 2025-12-10) — put `{"CDSS_API_KEY": "..."}` in a git-ignored `dwr_api.json` at the pipeline root.

In [ ]:
from climate_stations import fetch, audit, clean, config

# Sampling hooks — None means 'all in the seed' / 'all 12 measTypes'.
SITES = {"1886", "10417"}                 # CDSS stationNum; None for the full 40-station seed
MEAS_TYPES = {"Precip", "MaxTemp", "SnowSWE", "Solar"}  # None for all 12
print("API key configured:", bool(config.CDSS_API_KEY))

## 1 · Retrieve
Idempotent download of every `(station × measType)` series for its full period of record → immutable `data/original/` + sha256 manifest. A 404 = no-data (a station doesn't report every measure); errors are logged, not fatal.

In [ ]:
artifacts = fetch.fetch_all(sites=SITES, meas_types=MEAS_TYPES)
len(artifacts)

## 2 · Audit (raw)
Did the fetch return data? Profile `data/original/` before cleaning.

In [ ]:
print(audit.profile_raw())

## 3 · Cleanup
Ingest → tidy long → validate (pandera) → write `data/processed/climate-stations.csv` + `provenance.csv`. Units are harmonized **per concept** (degF / in / mJ/m^2 / kPa / KM); sub-zero temps are preserved, sentinels → NA.

In [ ]:
data, prov = clean.run(sites=SITES, meas_types=MEAS_TYPES, fail_on_empty=True)
data.head()

## 4 · Audit (processed)
Profile the deliverable: rows/ranges per variable, per-station coverage, the column profile, and the multi-network rollup (`network_summary` flags the NRCS/SNOTEL overlap with snowpack #11).

In [ ]:
print(audit.audit_processed())
audit.coverage_report()
audit.variables_report()
audit.network_summary()

## 5 · Publish / load
The local deliverable is `data/processed/climate-stations.csv`. Wide views are recipes (`docs/filter-pivot-recipes.md`); join the station dimension on `site_id`.

In [ ]:
import pandas as pd
df = pd.read_csv(config.CANONICAL_CSV, parse_dates=["datetime"], dtype={"site_id": str})
stations = pd.read_csv(config.STATIONS_CSV, dtype={"site_id": str})
df.merge(stations[["site_id", "network"]], on="site_id", how="left").head()